# Automatic Keyword Extractor — Google Colab
**Student:** Muhammed Fazal T M | **Register No.:** ASI23CA042

This notebook implements TF-IDF, RAKE, TextRank, YAKE, and KeyBERT, evaluates sample output, and launches an interactive Gradio demo.

**Instructions:** Open in Google Colab and choose **Runtime → Run all**. The first run downloads libraries and a language model and may take a few minutes.

In [ ]:
%pip -q install scikit-learn nltk rake-nltk yake keybert sentence-transformers spacy networkx pandas gradio
!python -m spacy download en_core_web_sm -q

In [ ]:
import re
import pandas as pd
import spacy
import networkx as nx
import yake
import gradio as gr
from sklearn.feature_extraction.text import TfidfVectorizer
from rake_nltk import Rake
from keybert import KeyBERT
from nltk.stem import PorterStemmer

nlp = spacy.load("en_core_web_sm")
keybert_model = KeyBERT(model="all-MiniLM-L6-v2")
stemmer = PorterStemmer()
print("Models loaded.")

In [ ]:
def preprocess_text(text):
    doc = nlp(text.lower())
    return " ".join(t.lemma_ for t in doc if not t.is_stop and not t.is_punct and not t.is_space and t.is_alpha)

def get_candidates(text, max_ngram=3):
    doc = nlp(text)
    candidates = set()
    for chunk in doc.noun_chunks:
        phrase = " ".join(t.lemma_.lower() for t in chunk if not t.is_stop and t.is_alpha)
        if phrase:
            candidates.add(phrase)
    for n in range(1, max_ngram + 1):
        for i in range(len(doc) - n + 1):
            chunk = list(doc[i:i+n])
            if all(t.is_alpha and not t.is_stop for t in chunk):
                candidates.add(" ".join(t.lemma_.lower() for t in chunk))
    return sorted(candidates)

def extract_tfidf(text, top_n=10):
    sentences = [preprocess_text(s.text) for s in nlp(text).sents]
    sentences = [s for s in sentences if s]
    if not sentences:
        return []
    try:
        vec = TfidfVectorizer(ngram_range=(1, 3), max_features=3000)
        mat = vec.fit_transform(sentences)
    except ValueError:
        return []
    scores = mat.sum(axis=0).A1
    return [(w, float(s)) for w, s in sorted(zip(vec.get_feature_names_out(), scores), key=lambda x:x[1], reverse=True)[:top_n]]

def extract_rake(text, top_n=10):
    r = Rake(language="english", min_length=1, max_length=3)
    r.extract_keywords_from_text(text)
    return [(p.lower(), float(s)) for s, p in r.get_ranked_phrases_with_scores()[:top_n]]

def extract_textrank(text, top_n=10):
    doc = nlp(text.lower())
    words = [t.lemma_.lower() for t in doc if t.is_alpha and not t.is_stop and t.pos_ in {"NOUN", "PROPN", "ADJ"}]
    graph = nx.Graph()
    for a, b in zip(words, words[1:]):
        if a != b:
            if graph.has_edge(a, b):
                graph[a][b]["weight"] += 1
            else:
                graph.add_edge(a, b, weight=1)
    if graph.number_of_nodes() == 0:
        return []
    scores = nx.pagerank(graph, weight="weight")
    ranked = [(p, sum(scores.get(w, 0.0) for w in p.split())) for p in get_candidates(text)]
    return sorted([(p,s) for p,s in ranked if s > 0], key=lambda x:x[1], reverse=True)[:top_n]

def extract_yake(text, top_n=10):
    ex = yake.KeywordExtractor(lan="en", n=3, top=top_n, dedupLim=0.7)
    return [(p.lower(), float(s)) for p,s in ex.extract_keywords(text)]

def extract_keybert(text, top_n=10):
    if not text.strip():
        return []
    return [(p.lower(), float(s)) for p,s in keybert_model.extract_keywords(
        text, keyphrase_ngram_range=(1,3), stop_words="english",
        top_n=top_n, use_mmr=True, diversity=0.5)]

METHODS = {
    "TF-IDF": extract_tfidf, "RAKE": extract_rake,
    "TextRank": extract_textrank, "YAKE": extract_yake,
    "KeyBERT": extract_keybert
}

def extract_all_keywords(text, top_n=10):
    columns = ["Method", "Rank", "Keyword / Keyphrase", "Score"]
    if not text or not text.strip():
        return pd.DataFrame(columns=columns)
    rows = []
    for method, fn in METHODS.items():
        try:
            for rank, (phrase, score) in enumerate(fn(text, int(top_n)), start=1):
                rows.append({"Method": method, "Rank": rank, "Keyword / Keyphrase": phrase, "Score": round(float(score), 4)})
        except Exception as e:
            rows.append({"Method": method, "Rank": "-", "Keyword / Keyphrase": f"Error: {e}", "Score": ""})
    return pd.DataFrame(rows)

In [ ]:
sample_text = """
Natural Language Processing is a branch of artificial intelligence that enables computers to understand human language.
Machine learning and deep learning are widely used in text classification, sentiment analysis, document summarization,
and information retrieval. Keyword extraction identifies important words and phrases from documents.
Transformer models such as BERT generate contextual representations that improve many NLP applications.
"""
display(extract_all_keywords(sample_text, 10))

In [ ]:
def normalize_keyword(phrase):
    return " ".join(stemmer.stem(w) for w in re.findall(r"[a-z0-9]+", str(phrase).lower()))

def evaluate_keywords(predicted, gold_keywords, k=5):
    pred = {normalize_keyword(x) for x in predicted[:k]} - {""}
    gold = {normalize_keyword(x) for x in gold_keywords} - {""}
    if not pred or not gold:
        return {"Precision": 0.0, "Recall": 0.0, "F1": 0.0}
    correct = len(pred & gold)
    precision, recall = correct / len(pred), correct / len(gold)
    f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
    return {"Precision": round(precision,4), "Recall": round(recall,4), "F1": round(f1,4)}

# Illustrative gold keywords only; replace with benchmark annotations for formal evaluation.
gold_keywords = ["natural language processing", "machine learning", "keyword extraction",
                 "transformer models", "BERT", "text classification", "information retrieval"]
results = extract_all_keywords(sample_text, 10)
evaluation_rows = []
for method in METHODS:
    predicted = results.loc[results.Method == method, "Keyword / Keyphrase"].tolist()
    for k in (5, 10):
        evaluation_rows.append({"Method": method, "K": k, **evaluate_keywords(predicted, gold_keywords, k)})
evaluation_df = pd.DataFrame(evaluation_rows)
display(evaluation_df)
evaluation_df.to_csv("evaluation_results.csv", index=False)
print("Saved evaluation_results.csv")

In [ ]:
def demo(text, top_n):
    return extract_all_keywords(text, int(top_n))

app = gr.Interface(
    fn=demo,
    inputs=[
        gr.Textbox(label="Paste an article or document", lines=12, value=sample_text),
        gr.Slider(minimum=5, maximum=20, value=10, step=1, label="Keywords per method")
    ],
    outputs=gr.Dataframe(label="Keyword extraction comparison"),
    title="Automatic Keyword Extractor",
    description="Compare TF-IDF, RAKE, TextRank, YAKE, and KeyBERT."
)
app.launch(share=True)

## Notes
- The gold keywords above are illustrative, not a benchmark result. Use manually annotated benchmark keywords for a proper evaluation.
- YAKE uses **lower-is-better** scores; other methods generally rank higher scores first. Scores are not directly comparable across methods.
- KeyBERT may take extra time on the first run while its model downloads.
- The Gradio launch cell creates a temporary share link for the demo.